# External model — streaming completion

OpenAI-compatible **`POST …/v1/chat/completions`** on the gateway host with **`stream: true`**. In theory these routes return **OpenAI-style** streaming: **`text/event-stream`** with **`data:`** lines whose JSON has **`choices[].delta`** (token text in **`content`** or **`text`**) and a final **`[DONE]`** (same idea as the public OpenAI streaming API). If the gateway ever deviates, the stream cell prints a **note** when it cannot extract text from that shape.

**Prereq:** Python 3.9+ (stdlib: `json`, `ssl`, `urllib`).

**Credentials:** Same pattern as **`maas-validation-demo-with-key.ipynb`**: `MAAS_API_KEY` / `API_KEY` / **Demo quick swap** for the bearer token (mint with `POST …/maas-api/v1/api-keys` and subscription **`external-models-subscription`** as in your shell script). **`VERIFY_TLS`** from the environment (`1` / `true` for strict TLS).

**Two presets:** **Box A** = GPT-4o, **Box B** = Claude. Set **`ACTIVE_EXTERNAL`** in **Demo quick swap** to **`"gpt4o"`** or **`"claude"`**.

## Demo quick swap

Set **`ACTIVE_EXTERNAL`** (and optionally **`DEMO_API_KEY`** between takes). Run this cell, then **Load presets**.

| Variable | Effect |
|----------|--------|
| `ACTIVE_EXTERNAL` | **`"gpt4o"`** or **`"claude"`** — which preset to use. |
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY` env. |

In [ ]:
ACTIVE_EXTERNAL = "claude"  # or "gpt4o"
DEMO_API_KEY = ""

## Load presets

**HOST**, routes, and model ids are fixed here — only **`ACTIVE_EXTERNAL`** in **Demo quick swap** selects GPT-4o vs Claude.

In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request

# Gateway host (same idea as HOST=maas.apps.... in curl)
HOST = "maas.apps.rosa.jland.li5b.p3.openshiftapps.com"

# --- Box A: GPT-4o ---
_PRESET_GPT4O = {
    "url": f"https://{HOST}/gpt-4o/v1/chat/completions",
    "model": "gpt-4o",
}

# --- Box B: Claude ---
_PRESET_CLAUDE = {
    "url": f"https://{HOST}/claude-sonnet-4-20250514/v1/chat/completions",
    "model": "claude-sonnet-4-20250514",
}

_presets = {"gpt4o": _PRESET_GPT4O, "claude": _PRESET_CLAUDE}
_ae = globals().get("ACTIVE_EXTERNAL", "gpt4o")
ACTIVE_EXTERNAL = (_ae.strip() if isinstance(_ae, str) and _ae.strip() else "gpt4o")
if ACTIVE_EXTERNAL not in _presets:
    raise SystemExit('ACTIVE_EXTERNAL must be "gpt4o" or "claude" (set in Demo quick swap)')
_sel = _presets[ACTIVE_EXTERNAL]
EXTERNAL_CHAT_COMPLETIONS_URL = _sel["url"]
EXTERNAL_MODEL_NAME = _sel["model"]

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")

print("ACTIVE_EXTERNAL :", ACTIVE_EXTERNAL)
print("Model id          :", EXTERNAL_MODEL_NAME)
print("POST URL          :", EXTERNAL_CHAT_COMPLETIONS_URL)
print("VERIFY_TLS        :", VERIFY_TLS)
print("API key set       :", bool(API_KEY))

## Stream one chat completion

Sends **`messages`**, **`stream: true`**, reads **OpenAI-style** SSE (`data:` lines with **`choices[].delta`**). Falls back to a single JSON object with **`choices[].message.content`** (non-stream). If HTTP **200** arrives but no text is parsed, a **note** is printed—the response may not match the OpenAI streaming shape above.

In [ ]:
USER_MESSAGE = "Say hello in one short sentence."
MAX_TOKENS = 256


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def _text_from_delta(delta):
    """OpenAI-style delta.content / delta.text, or list-shaped content blocks."""
    if not isinstance(delta, dict):
        return ""
    parts = []
    for key in ("content", "text"):
        v = delta.get(key)
        if isinstance(v, str) and v:
            parts.append(v)
        elif isinstance(v, list):
            for block in v:
                if isinstance(block, dict):
                    t = block.get("text") or block.get("content")
                    if isinstance(t, str):
                        parts.append(t)
                elif isinstance(block, str):
                    parts.append(block)
    return "".join(parts)


def _text_from_choice(ch):
    if not isinstance(ch, dict):
        return ""
    parts = []
    d = ch.get("delta")
    if isinstance(d, dict):
        parts.append(_text_from_delta(d))
    if isinstance(ch.get("text"), str) and ch["text"]:
        parts.append(ch["text"])
    msg = ch.get("message")
    if isinstance(msg, dict) and isinstance(msg.get("content"), str) and msg["content"]:
        parts.append(msg["content"])
    return "".join(parts)


def stream_chat_completions(
    url: str,
    *,
    api_key: str,
    model: str,
    user_message: str,
    max_tokens: int,
):
    """Stream tokens to stdout. Returns (http_status, error_or_none, format_note).
    On success, error is None. format_note is set if HTTP 200 but no text matched OpenAI-style parsing.
    On failure, http_status may be None if no HTTP response (e.g. TLS/DNS)."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_message}],
        "max_tokens": max_tokens,
        "stream": True,
    }
    body = json.dumps(payload).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    ctx = _ssl_ctx()
    total_chars = 0
    last_usage = None
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=300) as resp:
            status = resp.getcode()
            body = b""
            while True:
                line = resp.readline()
                if not line:
                    break
                body += line
                s = line.decode("utf-8", errors="replace").strip()
                if not s or s.startswith(":"):
                    continue
                if not s.startswith("data:"):
                    continue
                payload = s[len("data:") :].lstrip()
                if payload == "[DONE]":
                    break
                try:
                    obj = json.loads(payload)
                except json.JSONDecodeError:
                    continue
                if isinstance(obj, dict) and "usage" in obj:
                    last_usage = obj.get("usage")
                for ch in obj.get("choices") or []:
                    t = _text_from_choice(ch)
                    if t:
                        print(t, end="", flush=True)
                        total_chars += len(t)
                if isinstance(obj, dict) and not obj.get("choices"):
                    td = obj.get("delta")
                    if isinstance(td, dict):
                        t = _text_from_delta(td)
                        if t:
                            print(t, end="", flush=True)
                            total_chars += len(t)
            if total_chars == 0 and body.strip():
                try:
                    obj = json.loads(body.decode("utf-8", errors="replace").strip())
                except json.JSONDecodeError:
                    obj = None
                if isinstance(obj, dict):
                    for ch in obj.get("choices") or []:
                        t = _text_from_choice(ch)
                        if t:
                            print(t, end="", flush=True)
                            total_chars += len(t)
    except urllib.error.HTTPError as e:
        err = e.read().decode("utf-8", errors="replace")
        if not err.strip():
            err = str(e)
        print("HTTP", e.code, err[:2000])
        return (e.code, err, None)
    except urllib.error.URLError as e:
        msg = str(e.reason) if getattr(e, "reason", None) else str(e)
        print("URL error:", msg)
        return (None, msg, None)
    except OSError as e:
        msg = str(e)
        print("Error:", msg)
        return (None, msg, None)
    print()
    print("---")
    print("Stream finished. Approx printed chars:", total_chars)
    if last_usage:
        print("usage:", last_usage)
    format_note = None
    if status == 200 and total_chars == 0 and body.strip():
        format_note = (
            "Note: HTTP 200 but no assistant text was parsed. These routes are expected to return OpenAI-style streaming "
            "(SSE data: lines with JSON choices[].delta and content/text, or one JSON object with choices[].message.content). "
            "If the gateway uses a different format, inspect the raw response."
        )
        print(format_note)
    return (status, None, format_note)


http_status, stream_error, openai_format_note = stream_chat_completions(
    EXTERNAL_CHAT_COMPLETIONS_URL,
    api_key=API_KEY,
    model=EXTERNAL_MODEL_NAME,
    user_message=USER_MESSAGE,
    max_tokens=MAX_TOKENS,
)
print("HTTP status:", http_status, "| error:" if stream_error else "| ok", stream_error or "")